In [1]:
!pip install transformers datasets accelerate -q

In [2]:
import torch, math
from transformers import (
    GPT2LMHeadModel, GPT2Tokenizer, Trainer,
    TrainingArguments, DataCollatorForLanguageModeling, set_seed
)
from datasets import Dataset

set_seed(42)

In [3]:
def generate_text(model, tokenizer, prompt, max_length=100):
    model.eval()
    inputs = tokenizer.encode(prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        output = model.generate(
            inputs,
            max_length=max_length,
            temperature=0.8,
            top_k=50,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [4]:
tokenizer2 = GPT2Tokenizer.from_pretrained('gpt2')
model2 = GPT2LMHeadModel.from_pretrained('gpt2')

tokenizer2.pad_token = tokenizer2.eos_token
model2.config.pad_token_id = tokenizer2.eos_token_id

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [5]:
recipe_prompts = [
    'To make butter chicken',
    'For pasta carbonara',
    'To prepare a chocolate cake',
]

baseline2 = {}
for p in recipe_prompts:
    baseline2[p] = generate_text(model2, tokenizer2, p)
    print(f'Prompt: {p}\nOutput: {baseline2[p]}\n')

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Prompt: To make butter chicken
Output: To make butter chicken, just throw a little at a time. Add the chicken to the bottom of a large Dutch oven. Boil until it reaches 180 degrees. Stir in the milk and spices. Place chicken and chicken stock in a large saucepan. Bring to a boil, then allow to cool. Remove from heat. Reduce heat to medium-low. Cover with a lid and allow to cool. Cook in pan for 5 minutes, or until mixture is soft. Remove from pan, and remove

Prompt: For pasta carbonara
Output: For pasta carbonara:

1 cup fresh, fresh marinara, chopped

1/2 cup granulated sugar

1 tbsp all-purpose flour

1 tsp baking powder

1/2 tsp baking soda

1 tsp cayenne

1 tbsp olive oil

Method:

Preheat oven to 425 degrees f.

Combine chopped tomatoes, olive oil, flour and baking powder in a mixing bowl. Pour baking powder into the tomato

Prompt: To prepare a chocolate cake
Output: To prepare a chocolate cake, the top of the pan has to be the same size as the cake-like consistency.

I know the

In [6]:
recipes = [
    'to make butter chicken start by marinating chicken pieces in yogurt with turmeric chili powder and garam masala for one hour.',
    'heat butter in a pan and fry onions until golden brown then add ginger garlic paste and cook for two minutes.',
    'add tomato puree and cook on low heat for ten minutes until the oil separates from the masala.',
    'add the marinated chicken and cook on medium heat for fifteen minutes until fully cooked.',
    'finish with fresh cream and kasuri methi and serve hot with naan or steamed rice.',
]

dataset2 = Dataset.from_dict({'text': recipes})

tok2 = dataset2.map(
    lambda x: tokenizer2(x['text'], truncation=True, max_length=128, padding='max_length'),
    batched=True,
    remove_columns=['text']
)

split2 = tok2.train_test_split(test_size=0.15, seed=42)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [7]:
collator2 = DataCollatorForLanguageModeling(tokenizer=tokenizer2, mlm=False)

args2 = TrainingArguments(
    output_dir='./gpt2-recipes',
    num_train_epochs=5,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    eval_strategy='epoch',
)

In [8]:
trainer2 = Trainer(
    model=model2,
    args=args2,
    train_dataset=split2['train'],
    eval_dataset=split2['test'],
    data_collator=collator2,
)

trainer2.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,No log,3.641339
2,No log,3.525786
3,No log,3.477454
4,No log,3.452084
5,No log,3.448524


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=10, training_loss=2.368851089477539, metrics={'train_runtime': 10.6047, 'train_samples_per_second': 1.886, 'train_steps_per_second': 0.943, 'total_flos': 1306460160000.0, 'train_loss': 2.368851089477539, 'epoch': 5.0})

In [9]:
eval2 = trainer2.evaluate()
print(f'Perplexity: {math.exp(eval2["eval_loss"]):.2f}')

print('\n=== FINE-TUNED RECIPES (After Fine-Tuning) ===')

for p in recipe_prompts:
    ft = generate_text(model2, tokenizer2, p)
    print(f'Prompt: {p}')
    print(f'  Baseline:   {baseline2[p][:120]}')
    print(f'  Fine-Tuned: {ft[:120]}\n')

Perplexity: 31.45

=== FINE-TUNED RECIPES (After Fine-Tuning) ===
Prompt: To make butter chicken
  Baseline:   To make butter chicken, just throw a little at a time. Add the chicken to the bottom of a large Dutch oven. Boil until i
  Fine-Tuned: To make butter chicken, add garlic powder and cook until golden brown. Add chicken and cook for about 10 minutes until c

Prompt: For pasta carbonara
  Baseline:   For pasta carbonara:

1 cup fresh, fresh marinara, chopped

1/2 cup granulated sugar

1 tbsp all-purpose flour

1 tsp ba
  Fine-Tuned: For pasta carbonara

1 cup olive oil

½ cloves garlic minced

2 cups diced tomatoes

4 tablespoons tomato paste

¼ tsp s

Prompt: To prepare a chocolate cake
  Baseline:   To prepare a chocolate cake, the top of the pan has to be the same size as the cake-like consistency.

I know the next t
  Fine-Tuned: To prepare a chocolate cake, whisk together the egg whites, flour, baking powder, baking soda, vanilla extract, salt and

